# Build APK — ORBIT (Google Colab)

Notebook này tự động:
1. Cài **Java 21**, **Node.js 20**, **Android SDK** (`platform-tools`, `android-34`, `build-tools;34.0.0`).
2. Clone mã nguồn từ GitHub (`spdatdd/Vibe-code-Rhythm-Game-Web`) và **loại bỏ hoàn toàn** các thư mục/tệp liên quan đến chế độ YouTube (`analysis/`, `server/`, `netlify/`, `server.mjs`, ...).
3. Đóng gói `public/` bằng **Capacitor** (giao diện sáng kiểu iPhone, tự động **2D** màn hình dọc / **3D Three.js** màn hình ngang, tắt copy/paste), gắn icon từ `assets/icon.png`, bật **Fullscreen Immersive** (ẩn thanh trạng thái & thanh điều hướng, hỗ trợ xoay dọc/ngang), và build `./gradlew assembleDebug`.
4. Tải file `ORBIT-debug.apk` về máy hoặc lưu lên Google Drive.

In [ ]:
# @title 1. Cài đặt Java 21, Node.js 20 và Android SDK
%%bash
set -e

echo "==> Cài Java 21 & công cụ..."
apt-get update -qq
apt-get install -y -qq openjdk-21-jdk unzip wget curl git > /dev/null

echo "==> Kiểm tra / cài Node.js 20..."
if ! node -v 2>/dev/null | grep -qE '^v(20|22|24|26)\.'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
  apt-get install -y -qq nodejs > /dev/null
fi
node -v
java -version

echo "==> Cài Android Command Line Tools..."
export ANDROID_HOME=/opt/android-sdk
if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
  mkdir -p "$ANDROID_HOME/cmdline-tools"
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
  unzip -q /tmp/cmdline-tools.zip -d "$ANDROID_HOME/cmdline-tools"
  mv "$ANDROID_HOME/cmdline-tools/cmdline-tools" "$ANDROID_HOME/cmdline-tools/latest"
  rm -f /tmp/cmdline-tools.zip
fi

yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" --licenses > /dev/null || true
"$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" "platform-tools" "platforms;android-34" "build-tools;34.0.0"
echo "==> Xong môi trường Android SDK!"

In [ ]:
# @title 2. Clone mã nguồn từ GitHub & xóa phần YouTube/Backend
%%bash
set -e
REPO_URL="https://github.com/spdatdd/Vibe-code-Rhythm-Game-Web.git"
BRANCH="main"

rm -rf /content/orbit-game
git clone --depth 1 -b "$BRANCH" "$REPO_URL" /content/orbit-game
cd /content/orbit-game

# Loại bỏ toàn bộ thư mục/tệp liên quan đến chế độ YouTube và server backend
rm -rf analysis server netlify .cache .tools server.mjs netlify.toml render.yaml Dockerfile Setup-Analysis.ps1 Start-Game.cmd
ls -la /content/orbit-game

In [ ]:
# @title 3. Cấu hình Capacitor Fullscreen + Icon & Build APK
%%bash
set -e
export JAVA_TOOL_OPTIONS="-Xlog:os+container=off"
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))
if [ -d "/usr/lib/jvm/java-21-openjdk-amd64" ]; then
  export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
fi
echo "JAVA_HOME=$JAVA_HOME"
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk
export PATH="$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools"

cd /content/orbit-game

# Chuẩn bị gói web sạch (chỉ lấy public/, không chèn thư mục YouTube/backend)
rm -rf www
cp -r public www

# Cài Capacitor và tạo cấu hình
npm install --no-audit --no-fund @capacitor/core @capacitor/cli @capacitor/android
cat > capacitor.config.json <<'EOF'
{
  "appId": "com.orbit.rhythmlab",
  "appName": "ORBIT",
  "webDir": "www",
  "android": {
    "allowMixedContent": false,
    "backgroundColor": "#f2f4f8"
  }
}
EOF

if [ ! -d "android" ]; then
  npx cap add android
else
  npx cap sync android
fi

echo "==> Tạo icon APK từ assets/icon.png..."
python3 -c "import PIL" 2>/dev/null || pip install -q pillow
python3 - <<'PY'
from pathlib import Path
from PIL import Image

src = Path('assets/icon.png')
if not src.exists():
    raise FileNotFoundError('Thiếu assets/icon.png trong repo!')
im = Image.open(src).convert('RGBA')
res = Path('android/app/src/main/res')
sizes = {'mdpi': 48, 'hdpi': 72, 'xhdpi': 96, 'xxhdpi': 144, 'xxxhdpi': 192}
fg_sizes = {'mdpi': 108, 'hdpi': 162, 'xhdpi': 216, 'xxhdpi': 324, 'xxxhdpi': 432}
for dpi, px in sizes.items():
    d = res / f'mipmap-{dpi}'
    d.mkdir(parents=True, exist_ok=True)
    resized = im.resize((px, px), Image.Resampling.LANCZOS)
    resized.save(d / 'ic_launcher.png', 'PNG', optimize=True)
    resized.save(d / 'ic_launcher_round.png', 'PNG', optimize=True)
for dpi, px in fg_sizes.items():
    d = res / f'mipmap-{dpi}'
    canvas = Image.new('RGBA', (px, px), (9, 13, 27, 255))
    inner = int(px * 0.68)
    icon = im.resize((inner, inner), Image.Resampling.LANCZOS)
    off = (px - inner) // 2
    canvas.paste(icon, (off, off), icon)
    canvas.save(d / 'ic_launcher_foreground.png', 'PNG', optimize=True)
print('Đã tạo xong toàn bộ icon Android.')
PY

echo "==> Bật chế độ Fullscreen Immersive & hỗ trợ xoay dọc (2D) / ngang (3D)..."
python3 - <<'PY'
from pathlib import Path
import re

manifest_path = Path('android/app/src/main/AndroidManifest.xml')
manifest = manifest_path.read_text(encoding='utf-8')
if 'android:screenOrientation' in manifest:
    manifest = re.sub(r'android:screenOrientation="[^"]+"', 'android:screenOrientation="sensor"', manifest, count=1)
else:
    manifest = manifest.replace('<activity', '<activity android:screenOrientation="sensor"', 1)
manifest_path.write_text(manifest, encoding='utf-8')

styles_path = Path('android/app/src/main/res/values/styles.xml')
styles = styles_path.read_text(encoding='utf-8')
extra_items = '''        <item name="android:windowFullscreen">true</item>
        <item name="android:windowLayoutInDisplayCutoutMode">shortEdges</item>
        <item name="android:statusBarColor">@android:color/transparent</item>
        <item name="android:navigationBarColor">@android:color/transparent</item>'''
if 'windowFullscreen' not in styles:
    styles = re.sub(r'(<style[^>]*AppTheme\.NoActionBar[^>]*>)', r'\1\n' + extra_items, styles, count=1)
styles_path.write_text(styles, encoding='utf-8')

main_activity = Path('android/app/src/main/java/com/orbit/rhythmlab/MainActivity.java')
main_activity.write_text('''package com.orbit.rhythmlab;

import android.os.Bundle;
import android.view.WindowManager;
import androidx.core.view.WindowCompat;
import androidx.core.view.WindowInsetsCompat;
import androidx.core.view.WindowInsetsControllerCompat;
import com.getcapacitor.BridgeActivity;

public class MainActivity extends BridgeActivity {
    @Override
    protected void onCreate(Bundle savedInstanceState) {
        super.onCreate(savedInstanceState);
        getWindow().addFlags(WindowManager.LayoutParams.FLAG_KEEP_SCREEN_ON);
        enableImmersiveFullscreen();
    }

    @Override
    public void onResume() {
        super.onResume();
        enableImmersiveFullscreen();
    }

    @Override
    public void onWindowFocusChanged(boolean hasFocus) {
        super.onWindowFocusChanged(hasFocus);
        if (hasFocus) {
            enableImmersiveFullscreen();
        }
    }

    private void enableImmersiveFullscreen() {
        WindowCompat.setDecorFitsSystemWindows(getWindow(), false);
        WindowInsetsControllerCompat controller = WindowCompat.getInsetsController(getWindow(), getWindow().getDecorView());
        if (controller != null) {
            controller.setSystemBarsBehavior(WindowInsetsControllerCompat.BEHAVIOR_SHOW_TRANSIENT_BARS_BY_SWIPE);
            controller.hide(WindowInsetsCompat.Type.systemBars());
        }
    }
}
''', encoding='utf-8')
print('Đã cấu hình Immersive Fullscreen cho MainActivity.java.')
PY

cd android
chmod +x gradlew
./gradlew assembleDebug

cp app/build/outputs/apk/debug/app-debug.apk /content/ORBIT-debug.apk
ls -lh /content/ORBIT-debug.apk

In [ ]:
# @title 4. Tải file APK về máy
from google.colab import files
files.download('/content/ORBIT-debug.apk')

In [ ]:
# @title 5. (Tùy chọn) Lưu file APK lên Google Drive
from google.colab import drive
import shutil
drive.mount('/content/drive')
shutil.copy('/content/ORBIT-debug.apk', '/content/drive/MyDrive/ORBIT-debug.apk')
print('Saved to Google Drive: /content/drive/MyDrive/ORBIT-debug.apk')